<a href="https://colab.research.google.com/github/aitoufkir-khadija2004/TP4_Iris-Recognition/blob/main/TP4_Iris_IITD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP4 — Reconnaissance d'Iris : Chaîne Biométrique Complète
### Dataset : IITD Iris Database

---

**Objectif :** Implémenter la chaîne biométrique iris complète :
1. **Prétraitement** — débruitage, égalisation, suppression des reflets
2. **Détection** — Canny + Transformée de Hough circulaire (pupille & iris)
3. **Normalisation** — Rubber Sheet de Daugman
4. **Encodage** — Filtre de Gabor → IrisCode binaire
5. **Matching** — Distance de Hamming inter/intra-classe

**Dataset :** [IITD Iris Database](https://www4.comp.polyu.edu.hk/~csajaykr/IITD/Database_Iris.htm)  
Contient 2240 images de 224 sujets (5 images par œil gauche + 5 droit), format BMP/PNG 320×240 px.

---

## ⚙️ 0. Installation & Configuration

In [1]:
# Installation des dépendances (à exécuter une seule fois)
!pip install opencv-python-headless scikit-image matplotlib numpy scipy tqdm pandas seaborn -q

In [2]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd
import seaborn as sns
from scipy.signal import convolve2d
from skimage import exposure, filters, morphology
from tqdm import tqdm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ---- Style global ----
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'font.family': 'DejaVu Sans'
})

print("✅ Imports OK")
print(f"   OpenCV   : {cv2.__version__}")
print(f"   NumPy    : {np.__version__}")

✅ Imports OK
   OpenCV   : 4.13.0
   NumPy    : 2.0.2


In [3]:
# ============================================================
#  CONFIGURATION GLOBALE — modifier selon votre arborescence
# ============================================================

# Racine du dataset IITD
# Structure attendue :  IITD_Database/
#                          ├── 001/  (subject id)
#                          │     ├── 01.bmp
#                          │     └── ...
#                          └── ...
DATASET_ROOT = Path("IITD_Database")   # ← adapter si besoin

# Résolution du rubber-sheet normalisé (θ × r)
NORM_THETA  = 360   # colonnes (angle)
NORM_RADIUS = 64    # lignes   (rayon)

# Résolution de l'IrisCode (après sous-échantillonnage des réponses Gabor)
CODE_COLS = 256

# Nombre max de sujets à traiter pour la démo (None = tous)
MAX_SUBJECTS = 20

# Répertoire de sortie
OUT_DIR = Path("output_tp4")
OUT_DIR.mkdir(exist_ok=True)

print("✅ Configuration chargée")

✅ Configuration chargée


## 📁 1. Chargement du Dataset IITD

In [4]:
def load_iitd_dataset(root: Path, max_subjects=None):
    """
    Parcourt l'arborescence IITD et retourne une liste de dicts :
    [{'subject': '001', 'eye': 'left', 'image_id': 0, 'path': Path(...)}]
    """
    records = []
    if not root.exists():
        print(f"⚠️  Dossier '{root}' introuvable.")
        print("    Téléchargez le dataset IITD et placez-le dans le répertoire courant.")
        print("    URL : https://www4.comp.polyu.edu.hk/~csajaykr/IITD/Database_Iris.htm")
        return records

    subjects = sorted([d for d in root.iterdir() if d.is_dir()])
    if max_subjects:
        subjects = subjects[:max_subjects]

    for subj_dir in subjects:
        sid = subj_dir.name
        images = sorted(subj_dir.glob("*.bmp")) + sorted(subj_dir.glob("*.png"))
        for idx, img_path in enumerate(images):
            records.append({
                'subject'  : sid,
                'image_id' : idx,
                'path'     : img_path
            })

    print(f"✅ Dataset chargé : {len(records)} images, "
          f"{len(set(r['subject'] for r in records))} sujets")
    return records


records = load_iitd_dataset(DATASET_ROOT, MAX_SUBJECTS)

# ---- Aperçu ----
if records:
    df_meta = pd.DataFrame(records).drop(columns='path')
    display(df_meta.head(10))

⚠️  Dossier 'IITD_Database' introuvable.
    Téléchargez le dataset IITD et placez-le dans le répertoire courant.
    URL : https://www4.comp.polyu.edu.hk/~csajaykr/IITD/Database_Iris.htm


In [5]:
# ---- Visualiser quelques images brutes ----
def show_raw_samples(records, n=6):
    sample = records[:n]
    fig, axes = plt.subplots(1, n, figsize=(3*n, 3))
    fig.suptitle("Exemples d'images brutes — Dataset IITD", fontweight='bold')
    for ax, rec in zip(axes, sample):
        img = cv2.imread(str(rec['path']))
        if img is None:
            ax.axis('off'); continue
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Sujet {rec['subject']}\nImg {rec['image_id']}",
                     fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(OUT_DIR / "00_raw_samples.png", bbox_inches='tight')
    plt.show()

if records:
    show_raw_samples(records)

---
## 🔧 2. Prétraitement

### Pipeline :
1. Conversion en niveaux de gris
2. Débruitage (filtre médian + gaussien)
3. Égalisation adaptative du contraste (CLAHE)
4. Détection et masquage des reflets spéculaires

In [6]:
def preprocess_iris(img_bgr: np.ndarray) -> dict:
    """
    Prétraite une image oculaire BGR.
    Retourne un dict avec toutes les étapes intermédiaires.
    """
    result = {'original': img_bgr.copy()}

    # --- Étape 1 : Niveaux de gris ---
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    result['gray'] = gray

    # --- Étape 2 : Débruitage ---
    # Médian pour les éventuels pixels impulsionnels
    denoised = cv2.medianBlur(gray, 3)
    # Gaussien léger pour lisser
    denoised = cv2.GaussianBlur(denoised, (5, 5), 1.0)
    result['denoised'] = denoised

    # --- Étape 3 : CLAHE (amélioration contraste local) ---
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised)
    result['enhanced'] = enhanced

    # --- Étape 4 : Détection des reflets spéculaires ---
    # Les reflets sont des zones très lumineuses (> seuil adaptatif)
    _, specular_mask = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    # Conserver uniquement les régions très brillantes (>200 pour BMP classiques)
    bright_thresh = max(200, int(np.percentile(gray, 98)))
    _, specular_mask = cv2.threshold(gray, bright_thresh, 255, cv2.THRESH_BINARY)

    # Dilater légèrement pour couvrir le halo du reflet
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    specular_mask = cv2.dilate(specular_mask, kernel, iterations=1)
    result['specular_mask'] = specular_mask

    # Inpainting sur l'image améliorée pour remplir les reflets
    inpainted = cv2.inpaint(enhanced, specular_mask, inpaintRadius=5,
                             flags=cv2.INPAINT_TELEA)
    result['inpainted'] = inpainted

    return result


# ---- Test sur la 1ère image ----
if records:
    img_bgr = cv2.imread(str(records[0]['path']))
    proc = preprocess_iris(img_bgr)

    stages = ['gray', 'denoised', 'enhanced', 'specular_mask', 'inpainted']
    titles = ['Niveaux de gris', 'Débruité (Med+Gauss)',
              'CLAHE', 'Masque reflets', 'Inpainting (final)']

    fig, axes = plt.subplots(1, 5, figsize=(17, 3.5))
    fig.suptitle("Pipeline de Prétraitement", fontweight='bold', fontsize=13)
    for ax, key, title in zip(axes, stages, titles):
        cmap = 'Reds' if key == 'specular_mask' else 'gray'
        ax.imshow(proc[key], cmap=cmap)
        ax.set_title(title, fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(OUT_DIR / "01_preprocessing.png", bbox_inches='tight')
    plt.show()

### 📝 Analyse du prétraitement

| Étape | Rôle | Justification |
|-------|------|---------------|
| **Niveaux de gris** | Réduction dimensionnelle | L'iris est une structure texturée ; la couleur n'apporte pas d'info discriminante |
| **Médian 3×3** | Suppression bruit impulsionnel | Conserve les bords contrairement au gaussien seul |
| **Gaussien 5×5, σ=1** | Lissage doux avant Canny | Évite les faux bords sur le bruit haute fréquence |
| **CLAHE** | Contraste adaptatif | Compense les illuminations inhomogènes fréquentes en IITD |
| **Inpainting TELEA** | Suppression reflets cornéens | Améliore la qualité du template normalisé |

> **Note IITD :** Les images sont capturées en conditions contrôlées, mais des reflets de la lampe IR et des variations d'éclairage inter-sujet justifient systématiquement CLAHE + inpainting.

---
## 🔵 3. Détection par Transformée de Hough Circulaire

### Stratégie :
- **Pupille** : petit cercle sombre, rayon ~ 15–50 px (IITD 320×240)
- **Iris** : grand cercle autour de la pupille, rayon ~ 70–130 px
- Détection séquentielle : pupille d'abord → contraindre la recherche d'iris autour

In [7]:
def detect_circles_hough(preprocessed: dict) -> dict:
    """
    Détecte la pupille et l'iris via Canny + HoughCircles.
    Retourne un dict avec les cercles et les visualisations.
    """
    img = preprocessed['inpainted']  # image prétraitée
    h, w = img.shape

    # ==================================================
    # A. Détection de la PUPILLE
    # ==================================================
    # Léger blur avant HoughCircles (recommandé par OpenCV)
    blurred = cv2.GaussianBlur(img, (7, 7), 2)

    pupil_circles = cv2.HoughCircles(
        blurred,
        cv2.HOUGH_GRADIENT,
        dp=1,
        minDist=w // 4,       # un seul cercle (pupille unique)
        param1=50,            # seuil haut Canny
        param2=25,            # seuil accumulateur (abaisser si manqué)
        minRadius=int(w * 0.05),
        maxRadius=int(w * 0.18)
    )

    if pupil_circles is None:
        return None  # échec détection

    # Prendre le cercle le plus fort
    px, py, pr = np.round(pupil_circles[0, 0]).astype(int)

    # ==================================================
    # B. Détection de l'IRIS
    # ==================================================
    # On centre la recherche autour de la pupille détectée
    iris_circles = cv2.HoughCircles(
        blurred,
        cv2.HOUGH_GRADIENT,
        dp=1,
        minDist=w // 4,
        param1=40,
        param2=20,
        minRadius=int(pr * 1.8),   # l'iris est plus grand que la pupille
        maxRadius=int(w * 0.48)
    )

    if iris_circles is None:
        # Fallback : estimer l'iris à 2.8× le rayon de la pupille
        ix, iy, ir = px, py, int(pr * 2.8)
    else:
        # Sélectionner le cercle dont le centre est le plus proche de la pupille
        candidates = np.round(iris_circles[0]).astype(int)
        dists = np.hypot(candidates[:, 0] - px, candidates[:, 1] - py)
        best = candidates[np.argmin(dists)]
        ix, iy, ir = best

    # ==================================================
    # C. Extraction des bords Canny pour visualisation
    # ==================================================
    edges = cv2.Canny(blurred, threshold1=30, threshold2=80)

    return {
        'pupil' : (px, py, pr),
        'iris'  : (ix, iy, ir),
        'edges' : edges,
        'image' : img
    }


# ---- Test ----
if records:
    detection = detect_circles_hough(proc)

    if detection:
        px, py, pr = detection['pupil']
        ix, iy, ir = detection['iris']
        print(f"✅ Pupille : centre=({px},{py}), rayon={pr} px")
        print(f"✅ Iris    : centre=({ix},{iy}), rayon={ir} px")
    else:
        print("⚠️  Détection échouée")

In [8]:
def visualize_detection(proc_dict: dict, det_dict: dict, title="Détection Hough"):
    """Visualise les cercles détectés superposés sur l'image."""
    img = det_dict['image']
    edges = det_dict['edges']
    px, py, pr = det_dict['pupil']
    ix, iy, ir = det_dict['iris']

    # Image couleur pour les annotations
    overlay = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    cv2.circle(overlay, (px, py), pr, (0, 0, 255), 2)    # rouge = pupille
    cv2.circle(overlay, (px, py), 2,  (0, 0, 255), 3)
    cv2.circle(overlay, (ix, iy), ir, (0, 255, 0), 2)    # vert  = iris

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle(title, fontweight='bold', fontsize=12)

    axes[0].imshow(proc_dict['inpainted'], cmap='gray')
    axes[0].set_title('Image prétraitée')

    axes[1].imshow(edges, cmap='gray')
    # Dessiner les cercles en surimpression sur les bords
    for ax_circle in [axes[1]]:
        circle_p = plt.Circle((px, py), pr, color='red',  fill=False, lw=1.5, ls='--')
        circle_i = plt.Circle((ix, iy), ir, color='lime', fill=False, lw=1.5, ls='--')
        ax_circle.add_patch(circle_p)
        ax_circle.add_patch(circle_i)
    axes[1].set_title('Bords Canny + cercles Hough')

    axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    axes[2].set_title('Résultat final\n(rouge=pupille, vert=iris)')

    for ax in axes:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(OUT_DIR / "02_hough_detection.png", bbox_inches='tight')
    plt.show()


if records and detection:
    visualize_detection(proc, detection)

In [9]:
# ---- Analyse de l'impact des seuils Canny ----
def canny_threshold_analysis(img: np.ndarray):
    """
    Visualise l'effet de différents couples de seuils Canny.
    """
    configs = [
        (20,  60,  "Bas/Bas : trop de bruit"),
        (30,  80,  "Bas/Moyen : recommandé iris"),
        (50, 150,  "Moyen/Haut : bords forts seulement"),
        (100, 200, "Haut/Haut : sous-détection"),
    ]
    blurred = cv2.GaussianBlur(img, (7, 7), 2)

    fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
    fig.suptitle("Impact des seuils Canny sur la détection des contours d'iris",
                 fontweight='bold')
    for ax, (t1, t2, label) in zip(axes, configs):
        edges = cv2.Canny(blurred, t1, t2)
        ax.imshow(edges, cmap='gray')
        ax.set_title(f"thresh=({t1},{t2})\n{label}", fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(OUT_DIR / "03_canny_analysis.png", bbox_inches='tight')
    plt.show()


if records:
    canny_threshold_analysis(proc['inpainted'])

### 📝 Discussion — Seuils Canny & Hyperparamètres Hough

**Seuils Canny :**

| Couple | Effet | Risque |
|--------|-------|--------|
| (20, 60) | Détecte trop de contours → paupières, cils, reflets | Faux positifs Hough |
| (30, 80) | Bon équilibre pour l'iris IITD | **Recommandé** |
| (50, 150) | Seuls bords très marqués (limbique+pupillaire) | Peut manquer l'iris si image sombre |
| (100, 200) | Sous-détection | Cercles non trouvés |

**Paramètres HoughCircles :**
- `param1` : seuil haut interne à Canny — baisser si images sombres
- `param2` : seuil accumulateur — plus petit = plus de cercles détectés (plus de faux positifs)
- `minDist` : évite les doublons (un seul iris par image)
- `minRadius/maxRadius` : contraindre à la plage physique pupille/iris

**Stratégie anti-faux-positifs paupières/sourcils :**
- Détecter la pupille en premier (région sombre et compacte, plus fiable)
- Contraindre la recherche d'iris par `minRadius = 1.8 × r_pupille`
- Vérifier que les deux centres sont proches (< 15 px pour IITD)

---
## 🌀 4. Normalisation Rubber Sheet (Daugman)

**Principe :** Reparamétriser l'anneau iris en coordonnées polaires normalisées :
$$I(r, \theta) \;\longrightarrow\; I_{norm}(\hat{r}, \theta), \quad \hat{r} \in [0,1], \; \theta \in [0, 2\pi[$$

La transformation ramène chaque point de l'anneau $[R_{pupille}, R_{iris}]$ à une grille rectangulaire indépendante de la taille de l'iris.

In [10]:
def rubber_sheet_normalize(img: np.ndarray,
                            pupil: tuple,
                            iris: tuple,
                            n_theta: int = NORM_THETA,
                            n_r: int = NORM_RADIUS) -> tuple:
    """
    Implémente le mapping polaire de Daugman.

    Paramètres
    ----------
    img    : image en niveaux de gris (prétraitée)
    pupil  : (cx, cy, r_pupille)
    iris   : (cx, cy, r_iris)
    n_theta: résolution angulaire (colonnes)
    n_r    : résolution radiale    (lignes)

    Retourne
    --------
    normalized : np.ndarray (n_r × n_theta, uint8)
    mask       : np.ndarray binaire (1 = zone fiable, 0 = occlusion)
    """
    px, py, pr = pupil
    ix, iy, ir = iris

    h, w = img.shape

    # Grille d'angles et de rayons normalisés
    theta_arr = np.linspace(0, 2 * np.pi, n_theta, endpoint=False)  # (n_theta,)
    r_arr     = np.linspace(0, 1, n_r)                               # (n_r,)

    # Interpolation bilinéaire vectorisée
    R, T = np.meshgrid(r_arr, theta_arr, indexing='ij')  # (n_r, n_theta)

    # Rayon réel pour chaque point
    r_real = pr + R * (ir - pr)   # (n_r, n_theta)

    # Coordonnées cartésiennes
    # Centre interpolé entre pupille et iris (décalage de l'axe optique)
    cx = px + R * (ix - px)
    cy = py + R * (iy - py)

    x_coords = cx + r_real * np.cos(T)  # (n_r, n_theta)
    y_coords = cy + r_real * np.sin(T)

    # Clamp aux dimensions de l'image
    x_int = np.clip(np.round(x_coords).astype(int), 0, w - 1)
    y_int = np.clip(np.round(y_coords).astype(int), 0, h - 1)

    # Échantillonnage
    normalized = img[y_int, x_int].astype(np.uint8)  # (n_r, n_theta)

    # ==================================================
    # Masque des zones non fiables
    # ==================================================
    mask = np.ones((n_r, n_theta), dtype=np.uint8)

    # 1. Occlusion par les paupières : supérieure (θ ≈ π/2) et inférieure (θ ≈ 3π/2)
    #    On masque les bandes angulaires correspondant à ±30° autour de 90° et 270°
    eyelid_mask_angles = (
        (theta_arr > np.pi * 0.35) & (theta_arr < np.pi * 0.65)   # paupière sup
    ) | (
        (theta_arr > np.pi * 1.35) & (theta_arr < np.pi * 1.65)   # paupière inf
    )
    mask[:, eyelid_mask_angles] = 0

    # 2. Zones hors de l'image après clamp
    out_of_bounds = (
        (x_coords < 0) | (x_coords >= w) |
        (y_coords < 0) | (y_coords >= h)
    )
    mask[out_of_bounds] = 0

    # 3. Zones très sombres ou très claires (reflets résiduels)
    shadow_mask = (normalized < 20) | (normalized > 235)
    mask[shadow_mask] = 0

    return normalized, mask


# ---- Test ----
if records and detection:
    norm_img, norm_mask = rubber_sheet_normalize(
        proc['inpainted'],
        detection['pupil'],
        detection['iris']
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    fig.suptitle("Normalisation Rubber Sheet de Daugman", fontweight='bold')

    axes[0].imshow(proc['inpainted'], cmap='gray')
    px, py, pr = detection['pupil']; ix, iy, ir = detection['iris']
    circle_p = plt.Circle((px,py), pr, color='red',  fill=False, lw=1.5)
    circle_i = plt.Circle((ix,iy), ir, color='lime', fill=False, lw=1.5)
    axes[0].add_patch(circle_p); axes[0].add_patch(circle_i)
    axes[0].set_title("Image source + cercles")

    axes[1].imshow(norm_img, cmap='gray', aspect='auto')
    axes[1].set_title(f"Iris normalisé\n({NORM_RADIUS}×{NORM_THETA} px)")
    axes[1].set_xlabel("θ  [0 → 2π]")
    axes[1].set_ylabel("r  [pupille→iris]")

    axes[2].imshow(norm_mask, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    axes[2].set_title("Masque fiabilité\n(vert=fiable, rouge=occlusion)")
    axes[2].set_xlabel("θ  [0 → 2π]")

    for ax in axes:
        ax.axis('on')
    plt.tight_layout()
    plt.savefig(OUT_DIR / "04_rubber_sheet.png", bbox_inches='tight')
    plt.show()

    # Sauvegarder le masque
    np.save(OUT_DIR / "mask_sample.npy", norm_mask)
    print(f"✅ Masque exporté : {norm_mask.sum()} pixels fiables / {norm_mask.size} total "
          f"({100*norm_mask.mean():.1f}%)")

---
## 🔐 5. Encodage — Filtre de Gabor → IrisCode Binaire

**Principe (Daugman 1993) :**  
Le filtre de Gabor 2D encode localement la *phase* de la texture de l'iris.
La réponse complexe $G(r,\theta)$ est seuillée à son signe imaginaire → bit 0 ou 1.

$$\text{IrisCode}(r,\theta) = \text{sign}\big[\text{Im}\big(G * I_{norm}\big)\big]$$

In [11]:
def gabor_kernel_2d(size: int, sigma: float, freq: float,
                    theta: float = 0) -> np.ndarray:
    """
    Génère un noyau de Gabor 2D (partie imaginaire = en quadrature de phase).
    """
    half = size // 2
    x, y = np.meshgrid(np.arange(-half, half+1), np.arange(-half, half+1))

    x_rot =  x * np.cos(theta) + y * np.sin(theta)
    y_rot = -x * np.sin(theta) + y * np.cos(theta)

    envelope = np.exp(-(x_rot**2 + y_rot**2) / (2 * sigma**2))
    # Partie réelle
    real_part = envelope * np.cos(2 * np.pi * freq * x_rot)
    # Partie imaginaire
    imag_part = envelope * np.sin(2 * np.pi * freq * x_rot)

    return real_part + 1j * imag_part


def encode_iris(normalized: np.ndarray,
                mask: np.ndarray,
                n_cols: int = CODE_COLS,
                freq: float = 0.1,
                sigma: float = 3.0,
                n_orientations: int = 2) -> tuple:
    """
    Produit un IrisCode binaire à partir de l'iris normalisé.

    Retourne
    --------
    iris_code  : np.ndarray (1D bool) — le code binaire
    code_mask  : np.ndarray (1D bool) — masque associé
    code_2d    : np.ndarray 2D pour visualisation
    """
    # Normaliser l'image entre [0,1]
    img_f = normalized.astype(np.float64) / 255.0

    all_bits  = []
    all_masks = []

    for k in range(n_orientations):
        theta_k = k * np.pi / n_orientations
        kernel = gabor_kernel_2d(size=11, sigma=sigma, freq=freq, theta=theta_k)

        # Filtrage (séparé réel/imaginaire)
        resp_real = convolve2d(img_f, kernel.real, mode='same', boundary='wrap')
        resp_imag = convolve2d(img_f, kernel.imag, mode='same', boundary='wrap')

        # IrisCode = signe de la partie imaginaire
        bits_2d  = (resp_imag >= 0).astype(np.uint8)  # (n_r, n_theta)

        # Sous-échantillonnage à n_cols colonnes
        step = max(1, normalized.shape[1] // n_cols)
        bits_sub  = bits_2d[:, ::step][:, :n_cols]      # (n_r, n_cols)
        mask_sub  = mask[:, ::step][:, :n_cols]

        all_bits.append(bits_sub)
        all_masks.append(mask_sub)

    # Empiler les réponses multi-orientations
    code_2d  = np.vstack(all_bits)    # (n_r*n_orient, n_cols)
    mask_2d  = np.vstack(all_masks)

    # Aplatir en 1D
    iris_code  = code_2d.ravel().astype(bool)
    code_mask  = mask_2d.ravel().astype(bool)

    return iris_code, code_mask, code_2d


# ---- Test ----
if records and detection:
    iris_code, code_mask, code_2d = encode_iris(norm_img, norm_mask)

    fig, axes = plt.subplots(1, 3, figsize=(15, 3))
    fig.suptitle("IrisCode — Encodage par Filtre de Gabor", fontweight='bold')

    axes[0].imshow(norm_img, cmap='gray', aspect='auto')
    axes[0].set_title("Iris normalisé")

    axes[1].imshow(code_2d, cmap='binary', aspect='auto')
    axes[1].set_title(f"IrisCode 2D\n({code_2d.shape[0]}×{code_2d.shape[1]} bits)")

    # Distribution des bits
    valid_bits = iris_code[code_mask]
    axes[2].bar(['0', '1'], [np.sum(~valid_bits), np.sum(valid_bits)],
                color=['#2196F3', '#FF5722'], edgecolor='black')
    axes[2].set_title(f"Distribution des bits\n({len(valid_bits)} bits valides)")
    axes[2].set_ylabel("Nombre de bits")

    for ax in axes:
        ax.axis('on')
    axes[0].axis('off')
    axes[1].axis('off')

    plt.tight_layout()
    plt.savefig(OUT_DIR / "05_iriscode.png", bbox_inches='tight')
    plt.show()

    print(f"✅ IrisCode : {len(iris_code)} bits total | "
          f"{code_mask.sum()} bits valides ({100*code_mask.mean():.1f}%)")

---
## 🎯 6. Matching — Distance de Hamming

**Définition :**
$$HD = \frac{\|(A \oplus B) \cap M_A \cap M_B\|}{\|M_A \cap M_B\|}$$

où $M_A$ et $M_B$ sont les masques de fiabilité des deux codes.

**Seuil de décision :**
- $HD < 0.32$ → **même iris** (identité)
- $HD \geq 0.32$ → **iris différents**

La distribution théorique pour deux iris indépendants suit une **loi binomiale** centrée sur HD ≈ 0.50.

In [12]:
def hamming_distance(code_a: np.ndarray, mask_a: np.ndarray,
                     code_b: np.ndarray, mask_b: np.ndarray) -> float:
    """
    Distance de Hamming fractionnelle entre deux IrisCodes, en tenant
    compte des masques de fiabilité.

    Retourne -1 si le nombre de bits valides est insuffisant (< 100).
    """
    # Masque combiné : uniquement les bits fiables dans LES DEUX codes
    valid = mask_a & mask_b

    n_valid = np.sum(valid)
    if n_valid < 100:
        return -1.0  # pas assez de bits pour décision fiable

    # XOR sur les bits valides
    diff = np.logical_xor(code_a[valid], code_b[valid])
    return float(np.sum(diff)) / n_valid


# ---- Pipeline complet pour une image ----
def process_image(img_path: str) -> tuple:
    """
    Traite une image end-to-end :
    image → (iris_code, code_mask)
    Retourne (None, None) en cas d'échec.
    """
    img = cv2.imread(str(img_path))
    if img is None:
        return None, None

    prep  = preprocess_iris(img)
    det   = detect_circles_hough(prep)
    if det is None:
        return None, None

    norm, mask = rubber_sheet_normalize(prep['inpainted'],
                                        det['pupil'], det['iris'])
    code, cmask, _ = encode_iris(norm, mask)
    return code, cmask


print("✅ Fonctions de matching définies")

✅ Fonctions de matching définies


In [13]:
# ============================================================
#  Construction de la base de données d'IrisCodes
# ============================================================
print("Construction de la base de données d'IrisCodes...")

db = []  # liste de {'subject', 'image_id', 'code', 'mask'}

if records:
    for rec in tqdm(records, desc="Traitement"):
        code, mask = process_image(rec['path'])
        if code is not None:
            db.append({
                'subject'  : rec['subject'],
                'image_id' : rec['image_id'],
                'code'     : code,
                'mask'     : mask
            })

    print(f"\n✅ Base de données : {len(db)} IrisCodes valides sur {len(records)} images")
else:
    print("⚠️  Pas d'images chargées — génération de données synthétiques pour la démo")

    # DONNÉES SYNTHÉTIQUES pour démontrer le pipeline sans dataset
    np.random.seed(42)
    N_SUBJECTS = 10
    N_IMG_PER  = 5
    CODE_LEN   = NORM_RADIUS * CODE_COLS * 2

    for s in range(N_SUBJECTS):
        # Template de base pour ce sujet (simule une identité)
        base_code = np.random.randint(0, 2, CODE_LEN).astype(bool)
        base_mask = np.random.rand(CODE_LEN) > 0.3
        for i in range(N_IMG_PER):
            # Légère variation intra-classe (bruit ~3%)
            noise = np.random.rand(CODE_LEN) < 0.03
            code  = base_code ^ noise
            db.append({'subject': f"{s+1:03d}", 'image_id': i,
                       'code': code, 'mask': base_mask.copy()})

    print(f"✅ {len(db)} IrisCodes synthétiques générés ({N_SUBJECTS} sujets)")

Construction de la base de données d'IrisCodes...
⚠️  Pas d'images chargées — génération de données synthétiques pour la démo
✅ 50 IrisCodes synthétiques générés (10 sujets)


In [14]:
# ============================================================
#  Calcul des distances de Hamming intra/inter-classe
# ============================================================
print("Calcul des distances de Hamming...")

intra_hd = []   # même sujet
inter_hd = []   # sujets différents

# Regrouper par sujet
from collections import defaultdict
by_subject = defaultdict(list)
for entry in db:
    by_subject[entry['subject']].append(entry)

subjects = list(by_subject.keys())

# Intra-classe : toutes les paires du même sujet
for sid, entries in by_subject.items():
    for i in range(len(entries)):
        for j in range(i+1, len(entries)):
            hd = hamming_distance(entries[i]['code'], entries[i]['mask'],
                                  entries[j]['code'], entries[j]['mask'])
            if hd >= 0:
                intra_hd.append(hd)

# Inter-classe : paires de sujets différents (sous-échantillonnage)
np.random.seed(0)
n_inter_pairs = min(500, len(subjects) * (len(subjects)-1) // 2)
for _ in range(n_inter_pairs):
    s1, s2 = np.random.choice(len(subjects), 2, replace=False)
    e1 = np.random.choice(by_subject[subjects[s1]])
    e2 = np.random.choice(by_subject[subjects[s2]])
    hd = hamming_distance(e1['code'], e1['mask'], e2['code'], e2['mask'])
    if hd >= 0:
        inter_hd.append(hd)

intra_hd = np.array(intra_hd)
inter_hd = np.array(inter_hd)

print(f"✅ Intra-classe : {len(intra_hd)} paires | "
      f"HD moy = {intra_hd.mean():.4f} ± {intra_hd.std():.4f}")
print(f"✅ Inter-classe : {len(inter_hd)} paires | "
      f"HD moy = {inter_hd.mean():.4f} ± {inter_hd.std():.4f}")

Calcul des distances de Hamming...
✅ Intra-classe : 100 paires | HD moy = 0.0580 ± 0.0015
✅ Inter-classe : 45 paires | HD moy = 0.4983 ± 0.0037


In [ ]:
# ============================================================
#  Visualisation des distributions + ROC
# ============================================================

THRESHOLD = 0.32  # seuil de décision classique

fig = plt.figure(figsize=(16, 5))
fig.suptitle("Analyse de Performance du Système Iris", fontweight='bold', fontsize=13)

# ---- Graphe 1 : Histogrammes ----
ax1 = fig.add_subplot(1, 3, 1)
bins = np.linspace(0, 0.7, 50)
ax1.hist(intra_hd, bins=bins, alpha=0.7, color='#2196F3',
         label=f'Intra-classe (n={len(intra_hd)})', density=True)
ax1.hist(inter_hd, bins=bins, alpha=0.7, color='#FF5722',
         label=f'Inter-classe (n={len(inter_hd)})', density=True)
ax1.axvline(THRESHOLD, color='black', lw=2, ls='--',
            label=f'Seuil = {THRESHOLD}')
ax1.set_xlabel('Distance de Hamming')
ax1.set_ylabel('Densité')
ax1.set_title('Distributions HD\nIntra vs Inter classe')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

# ---- Graphe 2 : Courbe ROC ----
ax2 = fig.add_subplot(1, 3, 2)
thresholds = np.linspace(0, 1, 200)
tpr_list, fpr_list = [], []
for t in thresholds:
    tpr = np.mean(intra_hd <= t)   # vrais positifs = intra < seuil
    fpr = np.mean(inter_hd <= t)   # faux positifs  = inter < seuil
    tpr_list.append(tpr)
    fpr_list.append(fpr)

# AUC (approximation par trapèzes)
auc = np.trapz(tpr_list, fpr_list)
if auc < 0: auc = -auc  # corriger si ordre inversé

ax2.plot(fpr_list, tpr_list, color='#4CAF50', lw=2,
         label=f'ROC (AUC = {auc:.3f})')
ax2.plot([0, 1], [0, 1], 'k--', lw=1, label='Aléatoire')
# Marquer le seuil choisi
idx_t = np.argmin(np.abs(np.array(thresholds) - THRESHOLD))
ax2.scatter(fpr_list[idx_t], tpr_list[idx_t], color='red', s=80, zorder=5,
            label=f'HD={THRESHOLD}')
ax2.set_xlabel('Taux de Faux Positifs (FAR)')
ax2.set_ylabel('Taux de Vrais Positifs (TAR)')
ax2.set_title('Courbe ROC')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# ---- Graphe 3 : FAR & FRR vs seuil ----
ax3 = fig.add_subplot(1, 3, 3)
far_list = np.array(fpr_list)                      # FAR = inter < seuil
frr_list = 1 - np.array(tpr_list)                  # FRR = intra > seuil

ax3.plot(thresholds, far_list * 100, color='#FF5722', lw=2, label='FAR (%)')
ax3.plot(thresholds, frr_list * 100, color='#2196F3', lw=2, label='FRR (%)')
ax3.axvline(THRESHOLD, color='black', lw=1.5, ls='--', label=f'Seuil={THRESHOLD}')

# EER = point où FAR ≈ FRR
eer_idx = np.argmin(np.abs(far_list - frr_list))
eer = (far_list[eer_idx] + frr_list[eer_idx]) / 2
ax3.scatter(thresholds[eer_idx], eer*100, color='green', s=80, zorder=5,
            label=f'EER ≈ {eer*100:.1f}%')

ax3.set_xlabel('Seuil de Hamming')
ax3.set_ylabel('Taux d\'erreur (%)')
ax3.set_title('FAR & FRR vs Seuil')
ax3.legend(fontsize=8)
ax3.grid(alpha=0.3)
ax3.set_ylim(0, 100)

plt.tight_layout()
plt.savefig(OUT_DIR / "06_matching_analysis.png", bbox_inches='tight')
plt.show()

print(f"\n📊 Résultats au seuil HD = {THRESHOLD} :")
print(f"   FAR  = {far_list[idx_t]*100:.2f}% (Taux de Faux Positifs)")
print(f"   FRR  = {frr_list[idx_t]*100:.2f}% (Taux de Faux Négatifs)")
print(f"   AUC  = {auc:.4f}")
print(f"   EER  ≈ {eer*100:.2f}%")

In [ ]:
# ============================================================
#  Démonstration : Comparaison de 2 paires d'iris
# ============================================================

def demo_comparison(entry_a: dict, entry_b: dict, label_a: str, label_b: str):
    """Affiche une comparaison visuelle entre deux entrées."""
    hd = hamming_distance(entry_a['code'], entry_a['mask'],
                          entry_b['code'], entry_b['mask'])
    decision = "✅ MÊME IRIS" if hd < THRESHOLD else "❌ IRIS DIFFÉRENTS"
    color = 'green' if hd < THRESHOLD else 'red'

    fig, axes = plt.subplots(1, 2, figsize=(10, 2.5))
    fig.suptitle(f"{label_a}  vs  {label_b}  →  HD = {hd:.4f}  —  {decision}",
                 fontweight='bold', color=color)

    for ax, entry, label in zip(axes,
                                 [entry_a, entry_b],
                                 [label_a, label_b]):
        # Reconstruire le code 2D pour visualisation
        code_2d = entry['code'][:NORM_RADIUS * CODE_COLS].reshape(NORM_RADIUS, -1)
        ax.imshow(code_2d[:, :CODE_COLS], cmap='binary', aspect='auto')
        ax.set_title(f"IrisCode — {label}", fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.show()
    return hd


# Cas 1 : même sujet, images différentes
if len(subjects) >= 2 and len(by_subject[subjects[0]]) >= 2:
    s = subjects[0]
    e1, e2 = by_subject[s][0], by_subject[s][1]
    print("--- Comparaison INTRA-CLASSE ---")
    demo_comparison(e1, e2, f"Sujet {s}, img 0", f"Sujet {s}, img 1")

# Cas 2 : sujets différents
if len(subjects) >= 2:
    e1 = by_subject[subjects[0]][0]
    e2 = by_subject[subjects[1]][0]
    print("\n--- Comparaison INTER-CLASSE ---")
    demo_comparison(e1, e2,
                    f"Sujet {subjects[0]}",
                    f"Sujet {subjects[1]}")

---
## 📊 7. Récapitulatif & Analyse Finale

In [ ]:
# ============================================================
#  Tableau récapitulatif des métriques
# ============================================================

stats = {
    'Métrique'          : [
        'Taille de l\'IrisCode',
        'Bits valides (moyenne)',
        'HD intra-classe (μ ± σ)',
        'HD inter-classe (μ ± σ)',
        'Séparation (Δμ)',
        'AUC ROC',
        'EER',
        f'FAR @ seuil {THRESHOLD}',
        f'FRR @ seuil {THRESHOLD}',
    ],
    'Valeur' : [
        f"{len(db[0]['code']) if db else 'N/A'} bits",
        f"{np.mean([m.sum() for m in [e['mask'] for e in db]]):.0f} bits" if db else 'N/A',
        f"{intra_hd.mean():.4f} ± {intra_hd.std():.4f}",
        f"{inter_hd.mean():.4f} ± {inter_hd.std():.4f}",
        f"{inter_hd.mean() - intra_hd.mean():.4f}",
        f"{auc:.4f}",
        f"{eer*100:.2f}%",
        f"{far_list[idx_t]*100:.2f}%",
        f"{frr_list[idx_t]*100:.2f}%",
    ]
}

df_stats = pd.DataFrame(stats)
print("\n" + "="*50)
print("   TABLEAU DE PERFORMANCE — TP4 IRIS IITD")
print("="*50)
display(df_stats.style.set_properties(**{'text-align': 'left'}))

In [ ]:
# ============================================================
#  Visualisation récapitulative de la chaîne complète
# ============================================================

if records and detection:
    fig, axes = plt.subplots(1, 6, figsize=(20, 3))
    fig.suptitle(
        "Chaîne Biométrique Iris Complète — IITD Dataset",
        fontweight='bold', fontsize=13
    )

    # 1. Image brute
    axes[0].imshow(cv2.cvtColor(proc['original'], cv2.COLOR_BGR2RGB))
    axes[0].set_title('① Image brute', fontsize=9)

    # 2. Prétraitée
    axes[1].imshow(proc['inpainted'], cmap='gray')
    axes[1].set_title('② Prétraitée\n(CLAHE + inpainting)', fontsize=9)

    # 3. Détection Hough
    overlay = cv2.cvtColor(proc['inpainted'], cv2.COLOR_GRAY2BGR)
    px, py, pr = detection['pupil']; ix, iy, ir = detection['iris']
    cv2.circle(overlay, (px,py), pr, (0,0,255), 2)
    cv2.circle(overlay, (ix,iy), ir, (0,255,0), 2)
    axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    axes[2].set_title('③ Hough\n(pupille + iris)', fontsize=9)

    # 4. Iris normalisé
    axes[3].imshow(norm_img, cmap='gray', aspect='auto')
    axes[3].set_title('④ Rubber Sheet\nnormalisé', fontsize=9)

    # 5. Masque
    axes[4].imshow(norm_mask, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    axes[4].set_title('⑤ Masque\nfiabilité', fontsize=9)

    # 6. IrisCode
    axes[5].imshow(code_2d, cmap='binary', aspect='auto')
    axes[5].set_title('⑥ IrisCode\n(Gabor)', fontsize=9)

    for ax in axes:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(OUT_DIR / "07_pipeline_summary.png", bbox_inches='tight', dpi=150)
    plt.show()

    print(f"\n✅ Tous les résultats sauvegardés dans : {OUT_DIR.resolve()}")

---
## 📝 Conclusion & Discussion

### Ce que nous avons implémenté

| Étape | Méthode | Paramètres clés |
|-------|---------|------------------|
| **Prétraitement** | Médian + Gaussien + CLAHE + Inpainting TELEA | clipLimit=2.0, tileGrid=8×8 |
| **Détection pupille** | HoughCircles GRADIENT | param1=50, param2=25, r∈[5%,18%] de la largeur |
| **Détection iris** | HoughCircles contrainte | minR = 1.8×r_pupille |
| **Normalisation** | Rubber Sheet de Daugman | 64×360 px |
| **Encodage** | Filtre de Gabor (phase) | σ=3, freq=0.1, 2 orientations |
| **Matching** | Distance de Hamming fractionnelle | Seuil=0.32 |

### Limitations & Pistes d'amélioration

- **Paupières** : le masquage angulaire fixe est une approximation — une détection active (ex. Daugman integro-differential pour les paupières) améliorerait le masque.
- **Vitesse** : `convolve2d` en Python pur est lent — passer à `cv2.filter2D` ou une implémentation FFT accélèrerait l'encodage.
- **Variété des filtres** : utiliser plusieurs fréquences de Gabor (ex. 0.05, 0.1, 0.2) et 4+ orientations augmente la longueur et la robustesse de l'IrisCode.
- **Alignement rotatif** : lors du matching, tester les décalages circulaires du code (±k colonnes) pour compenser la rotation de l'œil et prendre le minimum HD.

### Références
- Daugman, J. (1993). *High confidence visual recognition of persons by a test of statistical independence*. IEEE TPAMI.
- Kumar, A. & Passi, A. (2010). *Comparison and combination of iris matchers for reliable personal authentication*. Pattern Recognition. (IITD dataset)
- Masek, L. (2003). *Recognition of Human Iris Patterns for Biometric Identification*. UWA Technical Report.

---
*TP4 — Traitement d'Images | ENSIAS 2024-2025*